In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, clear_output
import ipywidgets as widgets

## 1. Load HDF5 File

In [6]:
# Load the HDF5 file
# file_path = "/home/xinhai/projects/lerobot-arena/IsaacLab-Arena/data/nvidia/Arena-G1-Loco-Manipulation-Task/arena_g1_loco_manipulation_dataset_generated_small.hdf5"
file_path = "/home/xinhai/projects/lerobot-arena/IsaacLab-Arena/data/nvidia/Arena-G1-Loco-Manipulation-Task/arena_g1_loco_manipulation_dataset_annotated.hdf5"
# file_path = "/home/xinhai/projects/lerobot-arena/IsaacLab-Arena/data/automoma/summit_franka_open_microwave_7221_setstate.hdf5"
file = h5py.File(file_path, "r")

## 2. Print HDF5 File Structure

In [7]:
def print_hdf5_structure(name, obj, indent=0):
    """Recursively print the structure of an HDF5 file"""
    prefix = "  " * indent
    if isinstance(obj, h5py.Group):
        print(f"{prefix}- {name} (group)")
        for key in obj.keys():
            print_hdf5_structure(key, obj[key], indent + 1)
    elif isinstance(obj, h5py.Dataset):
        print(f"{prefix}- {name} (dataset)")
        print(f"{prefix}  shape: {obj.shape}, dtype: {obj.dtype}")

print("HDF5 File Structure:")
print("===================")
for key in file.keys():
    print_hdf5_structure(key, file[key])

HDF5 File Structure:
- data (group)
  - demo_0 (group)
    - action (group)
      - base_height_cmd (dataset)
        shape: (855, 1), dtype: float32
      - left_eef_pos (dataset)
        shape: (855, 3), dtype: float32
      - left_eef_quat (dataset)
        shape: (855, 4), dtype: float32
      - navigate_cmd (dataset)
        shape: (855, 3), dtype: float32
      - right_eef_pos (dataset)
        shape: (855, 3), dtype: float32
      - right_eef_quat (dataset)
        shape: (855, 4), dtype: float32
      - torso_orientation_rpy_cmd (dataset)
        shape: (855, 3), dtype: float32
    - actions (dataset)
      shape: (855, 23), dtype: float32
    - initial_state (group)
      - articulation (group)
        - robot (group)
          - joint_position (dataset)
            shape: (1, 43), dtype: float32
          - joint_velocity (dataset)
            shape: (1, 43), dtype: float32
          - root_pose (dataset)
            shape: (1, 7), dtype: float32
          - root_velocity (da

## 3. Per-Frame RGB and Depth Visualization

In [4]:
import ipywidgets as widgets
from IPython.display import display

# 1. Get list of available demos
# The structure is data -> demo_X
if 'data' in file:
    demo_keys = sorted(list(file['data'].keys()), key=lambda x: int(x.split('_')[-1]) if '_' in x else x)
else:
    print("Error: 'data' group not found in HDF5 file.")
    demo_keys = []

print(f"Found {len(demo_keys)} demos: {demo_keys[:5]} ...")

Found 10 demos: ['demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4'] ...


## 4. Per-Frame Point Cloud Visualization

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def visualize_demo_data(demo_name, step):
    """
    Visualizes RGB cameras and Robot States for a specific demo and timestep.
    Includes a WARNING if actions are all zero.
    """
    if not demo_name:
        return

    # Navigate to specific demo group
    demo_group = file['data'][demo_name]
    
    # --- Prepare Data Groups ---
    camera_obs = demo_group['camera_obs']
    obs = demo_group['obs']
    
    # --- Setup Plot ---
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, height_ratios=[1.2, 1, 1])
    
    # --- 1. RGB Images ---
    cam_keys = ['ego_topdown_rgb', 'ego_wrist_rgb', 'fix_local_rgb']
    for i, cam_key in enumerate(cam_keys):
        ax = fig.add_subplot(gs[0, i])
        if cam_key in camera_obs:
            img = camera_obs[cam_key][step]
            ax.imshow(img)
            ax.set_title(cam_key)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center')
        ax.axis('off')

    # --- 2. Joint Positions (12 DOF) ---
    joint_pos = obs['joint_pos'][step]
    ax_joints = fig.add_subplot(gs[1, :])
    colors = ['#1f77b4'] * 12
    ax_joints.bar(range(12), joint_pos, color=colors, alpha=0.7)
    ax_joints.set_title(f'Joint Positions (12 DOF) - Step {step}')
    ax_joints.set_ylim(np.min(joint_pos)-0.1, np.max(joint_pos)+0.1) # Dynamic scaling
    ax_joints.grid(True, axis='y', alpha=0.3)

    # --- 3. EEF Position & Rotation ---
    eef_pos = obs['eef_pos'][step]
    eef_quat = obs['eef_quat'][step]
    
    ax_eef = fig.add_subplot(gs[2, 0])
    ax_eef.bar(['x', 'y', 'z'], eef_pos, color='#2ca02c', alpha=0.7)
    ax_eef.set_title('EEF Position (m)')
    ax_eef.grid(True, alpha=0.3)

    ax_quat = fig.add_subplot(gs[2, 1])
    ax_quat.bar(['qx', 'qy', 'qz', 'qw'], eef_quat, color='#d62728', alpha=0.7)
    ax_quat.set_title('EEF Rotation (Quaternion)')
    ax_quat.set_ylim(-1.1, 1.1)
    ax_quat.grid(True, alpha=0.3)

    # --- 4. Actions (WITH WARNING) ---
    actions = obs['actions'][step]
    
    ax_act = fig.add_subplot(gs[2, 2])
    
    # Check if actions are all zero
    if np.all(np.isclose(actions, 0, atol=1e-5)):
        # Plot ghost bars just to show scale
        ax_act.bar(range(7), actions, color='gray', alpha=0.1)
        # Add BIG RED WARNING
        ax_act.text(0.5, 0.5, "⚠️ WARNING:\nALL ACTIONS ZERO", 
                    color='red', fontsize=14, fontweight='bold', 
                    ha='center', va='center', transform=ax_act.transAxes,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='red', boxstyle='round'))
        status_color = 'red'
    else:
        # Normal Plot
        ax_act.bar(range(7), actions, color='#ff7f0e', alpha=0.7)
        status_color = 'black'

    ax_act.set_title('Actions (7 Dim)', color=status_color)
    ax_act.set_ylim(-1.0, 1.0) # Fixed scale to see small movements
    ax_act.grid(True, alpha=0.3)

    # Final Title Update
    fig.suptitle(f'{demo_name} | Step {step}', fontsize=16)
    plt.tight_layout()
    plt.show()

# --- Re-run Widget Setup ---
if 'data' in file:
    demo_keys = sorted(list(file['data'].keys()), key=lambda x: int(x.split('_')[-1]) if '_' in x else x)
    
    demo_dropdown = widgets.Dropdown(
        options=demo_keys,
        value=demo_keys[0],
        description='Demo:',
        layout=widgets.Layout(width='300px')
    )

    init_len = file['data'][demo_keys[0]]['obs']['joint_pos'].shape[0]
    step_slider = widgets.IntSlider(
        value=0, 
        min=0, 
        max=init_len - 1, 
        description='Step:', 
        layout=widgets.Layout(width='600px')
    )

    def on_demo_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            new_len = file['data'][change['new']]['obs']['joint_pos'].shape[0]
            step_slider.max = new_len - 1
            step_slider.value = 0

    demo_dropdown.observe(on_demo_change)
    
    ui = widgets.HBox([demo_dropdown, step_slider])
    out = widgets.interactive_output(visualize_demo_data, {'demo_name': demo_dropdown, 'step': step_slider})
    
    display(ui, out)

Output()